# Advanced Product Search and Optimization with RedisVL

## Scenario

You are improving product search for an ecommerce catalog. The goal is not only to run a query. The goal is to understand the retrieval workflow well enough to explain, measure, and tune it.

In this 60-minute workshop you will:

- Prepare a judged product-search slice from the [WANDS dataset](https://github.com/wayfair/WANDS).
- Turn product metadata into `search_text` and embeddings.
- Load products into a Redis Search index with RedisVL.
- Test vector, filtered vector, `FT.HYBRID`, faceting, numeric filtering, and SQL-like query patterns.
- Score retrieval choices with nDCG@10, Recall@25, and query time.
- Finish with a practical recommendation for the next production experiment.

The live path uses a small judged WANDS sample. Full WANDS mode exists for validation and longer benchmark-style runs, not for the one-hour delivery.


## 1. Set Up the Product Search Data

The workshop has three pieces of data. Keep these roles separate as you work through the notebook:

| Artifact | What it contains | Why it matters |
|---|---|---|
| `corpus` | Product records from WANDS, including names, categories, descriptions, features, and more | This is what Redis will search over. |
| `queries` | Shopper search phrases | These are the inputs we test against. |
| `qrels` | Human relevance judgments for query/product pairs | These let us measure quality instead of guessing by eye. |

The prep script maps WANDS labels to scores: `Exact = 2`, `Partial = 1`, and `Irrelevant = 0`.

A WANDS query can have many relevant products. That is realistic for ecommerce, but it changes how you read recall: retrieving 25 products cannot recover hundreds or thousands of positives. The notebook will show that denominator before scoring any retrieval method.


### 1.1 Workshop Environment Settings

These values control the Redis URL, dataset mode, sample size, embedding model, and run-scoped Redis names. The defaults come from `.env.example`, and every Redis client is built from `REDIS_URL`.


In [ ]:
from pathlib import Path
import os
import re

from dotenv import load_dotenv

ROOT = Path('.').resolve()
load_dotenv(ROOT / '.env')

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
SAMPLE_PRODUCT_COUNT = int(os.getenv('SAMPLE_PRODUCT_COUNT', '600'))
SAMPLE_QUERY_COUNT = int(os.getenv('SAMPLE_QUERY_COUNT', '24'))
REDIS_LOAD_BATCH_SIZE = int(os.getenv('REDIS_LOAD_BATCH_SIZE', '25'))
EMBEDDING_CHUNK_SIZE = int(os.getenv('EMBEDDING_CHUNK_SIZE', '1024'))
EMBEDDING_BATCH_SIZE = int(os.getenv('EMBEDDING_BATCH_SIZE', '64'))
EVAL_QUERY_LIMIT_RAW = os.getenv('EVAL_QUERY_LIMIT', '').strip()
WORKSHOP_DATASET = os.getenv('WORKSHOP_DATASET', 'full').strip().lower()
if WORKSHOP_DATASET not in {'sample', 'full'}:
    raise ValueError("WORKSHOP_DATASET must be 'sample' or 'full'")
HF_MODEL = os.getenv('HF_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')
WORKSHOP_RUN_ID = re.sub(r'[^A-Za-z0-9_]', '_', os.getenv('WORKSHOP_RUN_ID', 'local')).strip('_') or 'local'

print('Workshop configuration')
print('- Redis URL: loaded from environment')
print(f'- Dataset split: {WORKSHOP_DATASET}')
print(f'- Embedding model: {HF_MODEL}')
print(f'- Workshop run ID: {WORKSHOP_RUN_ID}')
print(f"- Search study query limit: {EVAL_QUERY_LIMIT_RAW or 'auto'}")

if WORKSHOP_DATASET == 'sample':
    print(f'- Requested sample products: {SAMPLE_PRODUCT_COUNT:,}')
    print(f'- Requested sample queries: {SAMPLE_QUERY_COUNT:,}')
else:
    print('- Full mode: all WANDS products and loaded queries are used unless limited.')
    print(f'- Redis load batch size: {REDIS_LOAD_BATCH_SIZE:,}')
    print(f'- Embedding chunk size: {EMBEDDING_CHUNK_SIZE:,}')
    print(f'- Embedding batch size: {EMBEDDING_BATCH_SIZE:,}')

### 1.2 Operational Notes

Start Redis before the RedisVL sections so the notebook can create the index, cache embeddings, and run queries.

`WORKSHOP_RUN_ID` scopes the Redis index name, key prefix, and embedding cache names. Keep the default `local` for a single laptop. In a classroom or shared Redis instance, each user should have a unique value such as initials or a seat number.

The index-load cell later uses `overwrite=True, drop=True` because this is a repeatable workshop. That recreates the workshop index for the current run ID; do not point these names at a production index.

The sample is deterministic and query-first. It prioritizes judged queries and keeps at least one relevant product per query when the product budget allows it, so the default 600-product sample has enough queries for evaluation. The full dataset is best for realistic queries and search tuning.


### 1.3 Prepare the WANDS Dataset

The prep script downloads WANDS, validates the raw files, and writes the corpus, query, and qrels files. Use `WORKSHOP_DATASET=sample` for a smaller data sample or `WORKSHOP_DATASET=full` for an end-to-end full WANDS dataset.


In [ ]:
from scripts.prep_wands import prepare_wands

manifest = prepare_wands(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    max_products=SAMPLE_PRODUCT_COUNT,
    max_queries=SAMPLE_QUERY_COUNT,
    full=WORKSHOP_DATASET == 'full',
)

source = manifest['source']
sample = manifest['sample']

print(f"Dataset: {manifest['dataset']}")
print(f"Source:  {source['homepage']}")
print(source['note'])
print()
print('Raw WANDS counts')
print(f"- Products:  {source['raw_counts']['products']:,}")
print(f"- Queries:   {source['raw_counts']['queries']:,}")
print(f"- Judgments: {source['raw_counts']['judgments']:,}")
print()

selected_files = manifest['files'][WORKSHOP_DATASET]

if WORKSHOP_DATASET == 'full':
    print('Selected full WANDS counts')
    print(f"- Products:  {int(selected_files['products']):,}")
    print(f"- Queries:   {int(selected_files['queries_count']):,}")
    print(f"- Judgments: {int(selected_files['qrels_count']):,}")
else:
    print('Workshop sample counts')
    print(f"- Products:  {sample['products']:,}")
    print(f"- Queries:   {sample['queries']:,}")
    print(f"- Judgments: {sample['qrels']:,}")

    if sample['queries'] < sample['requested_queries']:
        print()
        print('Sample note')
        print(manifest['sample_note'])


### 1.4 Load Corpus, Queries, and Judgments

Now load the prepared files into memory. Redis is not involved yet; this is just local data setup.


In [ ]:
import json

import pandas as pd

pd.set_option('display.max_colwidth', 90)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

corpus = read_json(selected_files['corpus'])
queries = read_json(selected_files['queries'])
qrels = read_json(selected_files['qrels'])

corpus_df = pd.DataFrame(corpus.values())
corpus_by_id = corpus
EVAL_QUERY_LIMIT = int(EVAL_QUERY_LIMIT_RAW) if EVAL_QUERY_LIMIT_RAW else (8 if WORKSHOP_DATASET == 'sample' else len(queries))

print(f"Loaded {len(corpus_df):,} products, {len(queries):,} queries, and {sum(len(items) for items in qrels.values()):,} judgments.")
print(f"Search study will use {min(EVAL_QUERY_LIMIT, len(queries)):,} queries.")

### 1.5 Inspect Judgment Density

Before ranking anything, inspect the relevance-label shape. This distribution is key to understanding metrics like `Recall@25`.

> For Example: If a query has **hundreds** of relevant products, returning the top 25 items puts a hard ceiling on recall metrics even for an ideal retriever. This is important to ground expectations and also pick the right metrics to use for your use case.


In [ ]:
positive_counts = pd.Series({
    qid: sum(score > 0 for score in rels.values())
    for qid, rels in qrels.items()
}, name='positive_products')

positive_denominator = positive_counts.astype(float).where(positive_counts > 0)
oracle_recall_at_25 = (positive_counts.clip(upper=25).astype(float) / positive_denominator).fillna(0.0)

print(f"Queries with more than 25 positive products: {(positive_counts > 25).sum():,}/{len(positive_counts):,}")
print(f"Mean oracle Recall@25 ceiling: {oracle_recall_at_25.mean():.3f}")

### 1.6 Pick One Query to Follow

We will use one readable query through the query-pattern section. That makes it easier to compare how each retrieval method behaves.


In [ ]:
from collections import Counter


def relevant_product_ids(qid):
    return [pid for pid, score in qrels[qid].items() if score > 0 and pid in corpus_by_id]


def teaching_filter_class(qid):
    classes = [corpus_by_id[pid]['product_class'] for pid in relevant_product_ids(qid)]
    nonblank_classes = [value for value in classes if value]
    return Counter(nonblank_classes or classes).most_common(1)[0][0]


def choose_demo_query():
    for preferred in ['writing desk', 'unique coffee tables', 'card table']:
        for qid, text in queries.items():
            if text == preferred and relevant_product_ids(qid):
                return qid
    return next(qid for qid in queries if relevant_product_ids(qid))


demo_qid = choose_demo_query()
demo_query = queries[demo_qid]

print(f"Demo query: {demo_query!r}  (query_id={demo_qid})")
print('First few workshop queries:')
for qid, text in list(queries.items())[:8]:
    relevant_count = len(relevant_product_ids(qid))
    print(f"- {qid}: {text!r} ({relevant_count} relevant products in the selected dataset)")

Let's also preview a few product records to understand what we are dealing with.


In [ ]:
preview_df = corpus_df[['product_id', 'product_name', 'product_class', 'search_text']].head(5).copy()
preview_df['search_text'] = preview_df['search_text'].str.slice(0, 180) + '...'
display(preview_df)

## 2. Generate and Cache Embeddings

Each product is embedded from `search_text`, which combines name, class, category hierarchy, description, and features. RedisVL's `EmbeddingsCache` caches repeat vectors in Redis so reruns are faster after the first model pass.

The cache name includes `WORKSHOP_RUN_ID`, which prevents one participant's embeddings from colliding with another participant's run in shared Redis. Cache keys are deterministic for the text and model, and `ttl=None` means they live until Redis evicts them or someone deletes them. For a one-hour workshop this is helpful; for production pipelines you would usually persist generated embeddings with an expiration policy as part of your data pipeline.


### 2.1 Connect to Redis

Redis is required for the rest of the lab. This ping catches connection problems before we spend time embedding products.

In [ ]:
from redis import Redis

redis_client = Redis.from_url(REDIS_URL)
redis_client.ping()

### 2.2 Build the Vectorizer

The vectorizer turns product text and query text into vectors. The cache avoids recomputing embeddings across notebook reruns.

In [ ]:
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')

from redisvl.extensions.cache.embeddings import EmbeddingsCache
from redisvl.utils.vectorize import HFTextVectorizer

HF_CACHE_NAME = f'wands_hf_embeddings_{WORKSHOP_RUN_ID}'

embedding_cache = EmbeddingsCache(
    name=HF_CACHE_NAME,
    redis_url=REDIS_URL,
    ttl=None,
)

vectorizer = HFTextVectorizer(
    model=HF_MODEL,
    dtype='float32',
    cache=embedding_cache,
)

EMBEDDING_DIMS = vectorizer.dims
print(f'Using {HF_MODEL} with {EMBEDDING_DIMS} dimensions.')
print(f'Embedding cache: {HF_CACHE_NAME}')

### 2.3 Embed the Product Corpus

This adds one vector per product. Full WANDS runs can take several minutes, so the helper logs progress by outer chunk while the vectorizer handles model batches internally.


In [ ]:
def embed_texts_with_progress(texts, chunk_size, batch_size):
    embeddings = []
    total = len(texts)

    for start in range(0, total, chunk_size):
        chunk = texts[start:start + chunk_size]
        embeddings.extend(vectorizer.embed_many(
            chunk,
            batch_size=batch_size,
            as_buffer=True,
            normalize_embeddings=True,
        ))
        loaded = min(start + len(chunk), total)
        print(f'Embedded {loaded:,}/{total:,} products...')

    return embeddings


corpus_df['embedding'] = embed_texts_with_progress(
    corpus_df['search_text'].tolist(),
    chunk_size=EMBEDDING_CHUNK_SIZE,
    batch_size=EMBEDDING_BATCH_SIZE,
)

print(f"Embedded {len(corpus_df):,} products.")

## 3. Create the RedisVL Index

Before creating the index, underatand the separate the responsibilities:

| Layer | Role in this lab |
|---|---|
| Redis | Stores the product records as hashes and serves the index. |
| Redis Search | Provides indexed text, tag, numeric, vector, aggregation, and query execution capabilities. |
| RedisVL | Python tooling that defines schemas, loads records, builds query objects, manages embedding caches, and talks to Redis Search. |
| Redis Retrieval Optimizer | Runs repeatable retrieval experiments against a RedisVL index and reports quality and timing metrics. |

The index schema is the contract between product data and search behavior. Text fields support lexical search, tag and numeric fields support filters/facets, and the vector field supports semantic search.

### 3.1 Define the Index Schema

Keeping `search_text` and `embedding` in the same index lets one query combine exact terms, semantic similarity, and catalog constraints. For the live workshop we use a `FLAT` vector index because it is exact and easy to reason about on a small sample.

In [ ]:
INDEX_NAME = f'wands_products_{WORKSHOP_RUN_ID}'
INDEX_PREFIX = f'wands:product:{WORKSHOP_RUN_ID}'


def vector_attrs(algorithm='flat'):
    attrs = {
        'dims': EMBEDDING_DIMS,
        'distance_metric': 'cosine',
        'algorithm': algorithm,
        'datatype': 'float32',
    }

    if algorithm == 'hnsw':
        attrs.update({'m': 16, 'ef_construction': 200, 'ef_runtime': 20})
    elif algorithm == 'svs-vamana':
        attrs.update({
            'graph_max_degree': 40,
            'construction_window_size': 250,
            'search_window_size': 20,
            'compression': 'LVQ8',
        })

    return attrs

### 3.2 Assemble the Search Fields

The schema tells Redis Search which fields are text, tags, numbers, and vectors.

In [ ]:
def make_schema(algorithm='flat'):
    return {
        'index': {'name': INDEX_NAME, 'prefix': INDEX_PREFIX, 'storage_type': 'hash'},
        'fields': [
            {'name': 'product_id', 'type': 'tag'},
            {'name': 'product_name', 'type': 'text'},
            {'name': 'product_class', 'type': 'tag'},
            {'name': 'category_hierarchy', 'type': 'text'},
            {'name': 'search_text', 'type': 'text'},
            {'name': 'average_rating', 'type': 'numeric'},
            {'name': 'review_count', 'type': 'numeric'},
            {'name': 'embedding', 'type': 'vector', 'attrs': vector_attrs(algorithm)},
        ],
    }

### 3.3 Load Products into Redis

Each product becomes a Redis hash under the run-scoped `wands:product:<run-id>` prefix. Redis Search indexes those hashes using the schema above.

The `overwrite=True, drop=True` call is intentional for the workshop: rerunning the notebook recreates the current run's index from the prepared data. Use unique run IDs when multiple people share Redis.

Remote Redis targets can break large write pipelines. The loader uses modest retryable chunks so the same notebook works on local Redis and Redis Cloud.


In [ ]:
from redisvl.index import SearchIndex
import time


def redis_record(row):
    return {
        'product_id': row['product_id'],
        'product_name': row['product_name'],
        'product_class': row['product_class'],
        'category_hierarchy': row['category_hierarchy'],
        'search_text': row['search_text'],
        'average_rating': float(row['average_rating']),
        'review_count': int(row['review_count']),
        'embedding': row['embedding'],
    }


def load_with_retries(index, records, chunk_size, retries=3):
    loaded_keys = []
    total = len(records)

    for start in range(0, total, chunk_size):
        chunk = records[start:start + chunk_size]
        for attempt in range(1, retries + 1):
            try:
                loaded_keys.extend(index.load(chunk, id_field='product_id', batch_size=chunk_size))
                break
            except Exception:
                if attempt == retries:
                    raise
                time.sleep(2 * attempt)

        loaded = min(start + len(chunk), total)
        if loaded == total or loaded % 1000 < chunk_size:
            print(f'Loaded {loaded:,}/{total:,} products into Redis...')

    return loaded_keys

Create the index in Redis.

In [ ]:
index = SearchIndex.from_dict(make_schema('flat'), redis_url=REDIS_URL)
print(f'Creating index {INDEX_NAME!r} over key prefix {INDEX_PREFIX!r}.')
index.create(overwrite=True, drop=True)

Load the full dataset.

In [ ]:
records = [redis_record(row) for row in corpus_df.to_dict('records')]
loaded_keys = load_with_retries(index, records, chunk_size=REDIS_LOAD_BATCH_SIZE)

print(f"Loaded {len(loaded_keys):,} products into {INDEX_NAME}.")
index.info()

## 4. Product Search Patterns

Product search is hard because shoppers and catalogs rarely use the same language. A shopper may type `writing desk`; the catalog may say `study desk`, `desk and chair set`, or `office set`. No single retrieval method handles every case well, so production systems usually combine several signals.

These same retrieval patterns are also useful for recommendation candidate generation: find a broad candidate set quickly, then rerank or filter with more expensive business logic.


### 4.1 Patterns

The sequence includes vector search, filtered vector search, hybrid search, faceting, numeric filtering, and SQL-like lookup.

| Pattern | Product-search question | What to notice |
|---|---|---|
| Vector search | What products are semantically close to the query? | Useful when shopper words do not match catalog words exactly. |
| Filtered vector search | What semantic matches remain inside a category or constraint? | Filters narrow the candidate set before ranking. |
| Hybrid search | What happens when lexical and vector signals are combined server-side? | `FT.HYBRID` fuses text and vector scoring in one Redis command. |
| Faceting | What categories are available in the current corpus? | Facets support navigation and result refinement. |
| Numeric filter | What high-rated products match the query? | Numeric fields handle rating, price, inventory, and similar constraints. |
| `SQLQuery` | Can we express a business constraint query in SQL-like form? | Useful for readable text, tag, and numeric predicates. |


### 4.2 Failure Modes to Watch For

These are the tradeoffs students should look for in the outputs:

| Method | Can fail when... | Typical fix |
|---|---|---|
| Text search | The query uses different words than the catalog. | Add vector or hybrid retrieval. |
| Vector search | Semantic matches drift away from exact product constraints. | Add text signal, filters, or reranking. |
| Filters | The constraint is wrong or too narrow. | Let users control filters or use a trusted classifier/rule. |
| Hybrid search | The lexical/vector balance is off. | Tune the hybrid weight and evaluate with qrels. |

Vector distance is a ranking signal, not a business metric. Product quality still comes from relevance labels, business constraints, and user outcomes.

In [ ]:
RETURN_FIELDS = [
    'product_id',
    'product_name',
    'product_class',
    'average_rating',
    'vector_distance',
]


def show_results(rows):
    columns = [field for field in RETURN_FIELDS if rows and field in rows[0]]
    return pd.DataFrame(rows)[columns]


demo_vector = vectorizer.embed(demo_query, as_buffer=True, normalize_embeddings=True)

### 4.3 Vector Search

Vector search answers: what products are semantically close to the shopper query?

In [ ]:
from redisvl.query import VectorQuery


vector_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    return_fields=RETURN_FIELDS,
    num_results=5,
)

print(f"Vector search for: {demo_query!r}")
display(show_results(index.query(vector_query)))

### 4.4 Filtered Vector Search

Filtered vector search answers: what semantic matches remain after a catalog constraint is applied? For teaching, we use a known relevant class as a stand-in for a user-selected category.

In [ ]:
from redisvl.query.filter import Tag

filter_class = teaching_filter_class(demo_qid)
class_filter = Tag('product_class') == filter_class

filtered_vector_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    filter_expression=class_filter,
    return_fields=RETURN_FIELDS,
    num_results=5,
)

print(f"Filtered to product_class={filter_class!r}")
display(show_results(index.query(filtered_vector_query)))

### 4.5 Hybrid Search

RedisVL's `HybridQuery` sends `FT.HYBRID` to Redis, combining a text search leg and a vector similarity leg server-side. Here we use linear fusion. In Redis's linear method, `linear_alpha` is the text-score weight and `1 - linear_alpha` is the vector-score weight, so `0.65` means 65% text and 35% vector.


In [ ]:
from redisvl.query import HybridQuery


hybrid_query = HybridQuery(
    text=demo_query,
    text_field_name='search_text',
    vector=demo_vector,
    vector_field_name='embedding',
    combination_method='RRF',
    yield_combined_score_as='hybrid_score',
    return_fields=['product_id', 'product_name', 'product_class'],
    num_results=5,
)

display(pd.DataFrame(index.query(hybrid_query)))

### 4.6 Faceting

Faceting answers: what categories are available for navigation or refinement?

In [ ]:
from redis.commands.search.aggregation import AggregateRequest
from redis.commands.search.reducers import count

facet_query = (
    AggregateRequest('*')
    .group_by('@product_class', count().alias('count'))
    .limit(0, 100)
)

facet_rows = []
for row in index.aggregate(facet_query).rows:
    values = [item.decode('utf-8') if isinstance(item, bytes) else item for item in row]
    facet_rows.append(dict(zip(values[::2], values[1::2])))

facet_df = pd.DataFrame(facet_rows)
facet_df['count'] = facet_df['count'].astype(int)

display(facet_df.sort_values('count', ascending=False).head(10).reset_index(drop=True))


### 4.7 Numeric Filtering

Numeric filters handle constraints such as rating, price, inventory, distance, or freshness.

In [ ]:
from redisvl.query.filter import Num

rating_filter = Num('average_rating') >= 4
rating_query = VectorQuery(
    vector=demo_vector,
    vector_field_name='embedding',
    filter_expression=rating_filter,
    return_fields=RETURN_FIELDS,
    num_results=5,
)

display(show_results(index.query(rating_query)))

### 4.8 SQL-Like Query

`SQLQuery` is useful when a structured business question is clearer as SQL than as a query object. This example combines tokenized text search with numeric constraints; keep vector and hybrid ranking in the RedisVL query classes where those scoring controls are explicit.


In [ ]:
from redisvl.query import SQLQuery

sql = f"""
SELECT product_id, product_name, product_class, average_rating, review_count
FROM {INDEX_NAME}
WHERE fulltext(search_text, :query_text)
  AND average_rating >= :min_rating
  AND review_count >= :min_reviews
LIMIT 8
""".strip()

sql_params = {
    'query_text': demo_query,
    'min_rating': 3.5,
    'min_reviews': 1,
}

print(sql)
print(f"SQL fulltext query: {sql_query_text!r}")
display(pd.DataFrame(index.query(SQLQuery(sql, params=sql_params))))


## 5. Compare Tuning Choices

The live sample uses `FLAT` because it is exact and easy to reason about. Production work is different: you usually compare index type, query strategy, quality, latency, and memory before choosing.

The helper below includes example attributes for `HNSW` and `SVS-VAMANA`, but the core 60-minute path does not rebuild multiple indexes. Treat that as a follow-on benchmark once the evaluation workflow is clear.


### 5.1 Index and Query Strategy Menu

| Index type | Best use | Main tradeoff |
|---|---|---|
| `FLAT` | Small catalogs, exact evaluation, debugging | Can become slow or memory-heavy at larger scale. |
| `HNSW` | Large catalogs that need low-latency approximate search | Tune recall, memory, and build time. |
| `SVS-VAMANA` | Large vector sets where memory compression matters | Tune graph and compression settings. |

| Query strategy | Best use |
|---|---|
| Text | Exact terms, names, brands, SKUs, and attributes. |
| Vector | Semantic matches and vocabulary mismatch. |
| Hybrid | Lexical precision plus semantic recall. |
| Hybrid + filters | Business constraints such as category, rating, price, inventory, or policy. |

The schema helper already contains example attributes for each vector index family.

## 6. Optimize with WANDS Judgments

Now stop judging results by eye. WANDS qrels let us compare candidate retrieval strategies with the same queries and relevance labels. Redis Retrieval Optimizer is the owner of this experiment: each strategy is a named search method, the study runs every method, and the result is a comparable metrics table.

The important shift in this section is from "run one query and inspect rows" to "define an experiment contract." A search method says how to retrieve candidates; qrels say what good looks like; the study runner handles repetition, timing, scoring, and persistence.


### 6.1 Search Study Shape

A search study compares methods against an existing RedisVL index. We already did the expensive setup work: products are embedded, Redis hashes are loaded, and the Redis Search index exists. The optimizer should reuse that index rather than rebuild it.

| Concept | Role in this lab |
|---|---|
| `SearchMethodInput` | Carries the RedisVL index, queries, vectorizer, field names, and `ret_k`. |
| `SearchMethodOutput` | Returns a `ranx.Run` plus query timing metrics. |
| `search_method_map` | Binds names in the config to Python functions that build and run queries. |
| `run_search_study` | Executes every named method, evaluates qrels, persists study metrics, and returns a table. |

The useful extension point is `search_method_map`: we can register parameterized methods such as `hybrid_linear_text_020` or `hybrid_linear_text_080` without leaving the optimizer workflow. Each name becomes a row in the result table, which makes tuning choices explainable and repeatable.

The optimizer reports broad `ranx` metrics by default. After the study runs, we add explicit `nDCG@10`, `Recall@25`, and `Precision@25` from the same captured `ranx.Run` objects so the table matches the metrics introduced earlier in the workshop.


### 6.2 Shared Study Helpers

Redis Retrieval Optimizer expects each method to return ranked document IDs in `ranx.Run` format: `{query_id: {product_id: score}}`. RedisVL returns rows, so these helpers do three small translation jobs.

1. Normalize query text because the optimizer can pass plain strings or richer query records.
2. Normalize Redis result IDs because Redis clients may return bytes or strings.
3. Convert each result row into a score dictionary that `ranx` can evaluate.

The score values only need to preserve ranking order for these metrics. Text and hybrid methods return higher-is-better scores. Vector search returns a distance, so the vector method below converts distance into a higher-is-better similarity proxy.


In [ ]:
import pandas as pd

from ranx import Qrels, Run, evaluate
from redis_retrieval_optimizer.schema import SearchMethodOutput
from redis_retrieval_optimizer.search_methods.base import run_search_w_time
from redis_retrieval_optimizer.search_study import run_search_study
from redisvl.query import HybridQuery, TextQuery, VectorQuery


def query_text(raw_query):
    return raw_query['query'] if isinstance(raw_query, dict) else raw_query


def clean_doc_id(value):
    if isinstance(value, bytes):
        return value.decode('utf-8')
    return str(value) if value is not None else None


def rows_to_scores(rows, id_field_name, score_field, score_transform=lambda score: score):
    scores = {}
    for rank, row in enumerate(rows, start=1):
        product_id = clean_doc_id(row.get(id_field_name))
        if product_id is None:
            continue
        raw_score = row.get(score_field)
        scores[product_id] = 1.0 / rank if raw_score is None else score_transform(float(raw_score))
    return scores or {'no_match': 0.0}


### 6.3 Define Parameterized Search Methods

Each method receives the same `SearchMethodInput`: same index, same query set, same field names, same embedding model, and same `ret_k`. Only the RedisVL query object changes.

This is the clean way to tune retrieval behavior with Redis Retrieval Optimizer. Instead of writing a separate manual loop for every parameter, we create small named methods and register them in `study_search_method_map`. The study runner then treats `bm25_text`, `vector_cosine`, `hybrid_rrf`, and each linear hybrid weight as comparable candidates.


In [ ]:
def run_query_method(search_input, query_factory, score_field, score_transform=lambda score: score):
    ranked_results = {}

    for query_id, raw_query in search_input.raw_queries.items():
        query = query_factory(search_input, query_text(raw_query))
        rows = run_search_w_time(search_input.index, query, search_input.query_metrics)
        ranked_results[str(query_id)] = rows_to_scores(
            rows,
            id_field_name=search_input.id_field_name,
            score_field=score_field,
            score_transform=score_transform,
        )

    return SearchMethodOutput(
        run=Run(ranked_results),
        query_metrics=search_input.query_metrics,
    )


def bm25_text_method(search_input):
    return run_query_method(
        search_input,
        query_factory=lambda inputs, text: TextQuery(
            text=text,
            text_field_name=inputs.text_field_name,
            return_fields=[inputs.id_field_name, inputs.text_field_name],
            num_results=inputs.ret_k,
        ),
        score_field='score',
    )


def vector_method(search_input):
    def make_query(inputs, text):
        vector = inputs.emb_model.embed(text, as_buffer=True, normalize_embeddings=True)
        return VectorQuery(
            vector=vector,
            vector_field_name=inputs.vector_field_name,
            return_fields=[inputs.id_field_name, inputs.text_field_name],
            num_results=inputs.ret_k,
        )

    return run_query_method(
        search_input,
        make_query,
        'vector_distance',
        lambda distance: 1.0 - distance,
    )


In [ ]:
def make_hybrid_method(combination_method='LINEAR', text_weight=0.5):
    def hybrid_method(search_input):
        def make_query(inputs, text):
            vector = inputs.emb_model.embed(text, as_buffer=True, normalize_embeddings=True)
            query_kwargs = {
                'text': text,
                'text_field_name': inputs.text_field_name,
                'vector': vector,
                'vector_field_name': inputs.vector_field_name,
                'vector_search_method': 'KNN',
                'knn_ef_runtime': None,
                'combination_method': combination_method,
                'yield_combined_score_as': 'hybrid_score',
                'return_fields': [inputs.id_field_name, inputs.text_field_name],
                'num_results': inputs.ret_k,
            }
            if combination_method == 'LINEAR':
                query_kwargs['linear_alpha'] = text_weight
            else:
                query_kwargs['rrf_window'] = max(inputs.ret_k, 20)
                query_kwargs['rrf_constant'] = 60
            return HybridQuery(**query_kwargs)

        return run_query_method(search_input, make_query, 'hybrid_score')

    return hybrid_method


STUDY_METHODS = {
    'bm25_text': bm25_text_method,
    'vector_cosine': vector_method,
    'hybrid_rrf': make_hybrid_method('RRF'),
    'hybrid_linear_text_020': make_hybrid_method('LINEAR', text_weight=0.20),
    'hybrid_linear_text_035': make_hybrid_method('LINEAR', text_weight=0.35),
    'hybrid_linear_text_050': make_hybrid_method('LINEAR', text_weight=0.50),
    'hybrid_linear_text_065': make_hybrid_method('LINEAR', text_weight=0.65),
    'hybrid_linear_text_080': make_hybrid_method('LINEAR', text_weight=0.80),
}

study_outputs = {}


def capture_method(name, method):
    def wrapped(search_input):
        output = method(search_input)
        study_outputs[name] = output
        return output

    return wrapped


study_search_method_map = {
    name: capture_method(name, method)
    for name, method in STUDY_METHODS.items()
}


### 6.4 Prepare the Judged Query Set

The optimizer config takes file paths for queries and qrels. The full WANDS files are already on disk, but this workshop also supports a smaller query slice for pacing. This cell writes the exact slice used by the study so the run is reproducible and inspectable after the notebook finishes.

For sample mode, the default is the first eight judged queries unless `EVAL_QUERY_LIMIT` is set. For full mode, the default is every loaded WANDS query. The qrels are sliced to the same query IDs, which keeps evaluation honest: every method is scored against the same query set and the same relevance labels.


In [ ]:
study_query_ids = list(queries)[: min(EVAL_QUERY_LIMIT, len(queries))]
study_queries = {query_id: queries[query_id] for query_id in study_query_ids}
study_qrels = {query_id: qrels[query_id] for query_id in study_query_ids}

study_queries_path = PROCESSED_DIR / f'queries_{WORKSHOP_DATASET}_{WORKSHOP_RUN_ID}_study.json'
study_qrels_path = PROCESSED_DIR / f'qrels_{WORKSHOP_DATASET}_{WORKSHOP_RUN_ID}_study.json'

study_queries_path.write_text(json.dumps(study_queries, indent=2, sort_keys=True), encoding='utf-8')
study_qrels_path.write_text(json.dumps(study_qrels, indent=2, sort_keys=True), encoding='utf-8')

print(f'Search study will evaluate {len(study_queries):,} queries and {len(study_search_method_map):,} methods.')
print(f'- Queries: {study_queries_path}')
print(f'- Qrels:   {study_qrels_path}')


### 6.5 Run the Search Study

The config below tells Redis Retrieval Optimizer what to reuse and what to compare.

| Config field | What happens at execution time |
|---|---|
| `index_name` | Connects to the RedisVL index created earlier; it does not rebuild the index. |
| `queries`, `qrels` | Loads the exact judged query slice prepared in the previous cell. |
| `search_methods` | Runs the named methods registered in `study_search_method_map`. |
| `ret_k` | Retrieves 25 candidates per query for every method. |
| `id/text/vector_field_name` | Maps optimizer expectations to this product schema. |
| `embedding_model` | Recreates the same vectorizer and uses a run-scoped Redis embedding cache for query embeddings. |

During execution, the study loops over every method and every query, records query timings, builds a `ranx.Run`, scores it against qrels, and persists intermediate/final metrics at `study:<study_id>` in Redis. That persisted study key is useful for debugging and for comparing later runs with the same Redis database.

After `run_search_study` returns, the notebook merges in top-k metrics from the captured runs and sorts by `nDCG@10`, then `Recall@25`, then latency. That sorting reflects the product-search goal: first-page relevance first, candidate coverage second, speed as the tie-breaker.


In [ ]:
optimizer_config = {
    'study_id': f'wands-search-study-{WORKSHOP_RUN_ID}',
    'index_name': INDEX_NAME,
    'queries': str(study_queries_path),
    'qrels': str(study_qrels_path),
    'search_methods': list(study_search_method_map),
    'ret_k': 25,
    'id_field_name': 'product_id',
    'text_field_name': 'search_text',
    'vector_field_name': 'embedding',
    'embedding_model': {
        'type': 'hf',
        'model': HF_MODEL,
        'dim': EMBEDDING_DIMS,
        'embedding_cache_name': f'wands_retopt_embeddings_{WORKSHOP_RUN_ID}',
        'dtype': 'float32',
    },
}

optimizer_df = run_search_study(
    redis_url=REDIS_URL,
    config=optimizer_config,
    search_method_map=study_search_method_map,
)

qrels_obj = Qrels(study_qrels)
top_k_rows = []
for method_name, output in study_outputs.items():
    top_k_metrics = evaluate(
        qrels_obj,
        output.run,
        metrics=['ndcg@10', 'recall@25', 'precision@25'],
        make_comparable=True,
    )
    top_k_rows.append({'search_method': method_name, **top_k_metrics})

top_k_df = pd.DataFrame(top_k_rows)
optimizer_df = optimizer_df.merge(top_k_df, on='search_method', how='left')
optimizer_df['avg_query_ms'] = (optimizer_df['avg_query_time'] * 1000).round(3)
optimizer_df = optimizer_df.sort_values(
    ['ndcg@10', 'recall@25', 'avg_query_ms'],
    ascending=[False, False, True],
).reset_index(drop=True)

display(optimizer_df[[
    'search_method',
    'ndcg@10',
    'recall@25',
    'precision@25',
    'ndcg',
    'recall',
    'avg_query_ms',
    'ret_k',
]])


### 6.6 Result

The output below is the concrete recommendation from this run. It is not a universal claim that one method always wins; it is the best method for this data slice, this index, this embedding model, and this metric ordering.

In production, repeat the same workflow on a larger or fresher judged set, then expand `study_search_method_map` with filters, rerankers, `ret_k` variants, and index-type experiments.

Production checklist:

1. Keep product data fresh with an incremental indexing path, not only full reloads.
2. Use per-environment index names, prefixes, and cache names.
3. Monitor latency, memory, index size, embedding failures, and zero-result queries.
4. Evaluate on a larger judged slice before trusting a strategy change.
5. Roll out with an A/B test or shadow evaluation when user behavior is the final judge.


In [ ]:
best = optimizer_df.iloc[0]

print('Best method from the Redis Retrieval Optimizer search study')
print(f"- Search method: {best['search_method']}")
print(f"- nDCG@10: {round(float(best['ndcg@10']), 4)}")
print(f"- Recall@25: {round(float(best['recall@25']), 4)}")
print(f"- Precision@25: {round(float(best['precision@25']), 4)}")
print(f"- Average query time: {best['avg_query_ms']} ms")

print()
print(f"Persisted study key: study:{optimizer_config['study_id']}")
print('Next experiment: add filters, reranking, ret_k, and index-type variants to the search_method_map or a larger grid study.')
